# IFCNetCore — SigLIP 2 visual embeddings

Runs `google/siglip2-large-patch16-512` over all 12 stock renders per object,
mean-pools across views, writes one 1024-d `.npy` per object.

**Upload to your Drive:** `IFCNetCorePng.zip` (445 MB renders zip).
**Output to Drive:** `siglip_ifcnet_colorless.zip`.

Resume-safe: re-running skips obj_ids whose `.npy` already exists.

Use a fresh runtime to avoid package conflicts with the DuoDuo / DINOv3 notebooks.
Pick a GPU runtime first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
from pathlib import Path

# Adjust if your zip lives elsewhere
DRIVE_ZIP    = Path('/content/drive/MyDrive/IFCNetCorePng.zip')
DRIVE_OUT    = Path('/content/drive/MyDrive')

RENDERS_ROOT  = Path('/content/data/IFCNetCore/renders')
FEATURES_ROOT = Path('/content/data/IFCNetCore/processed/rendered_features')
METADATA_PATH = Path('/content/data/IFCNetCore/metadata.json')

assert DRIVE_ZIP.exists(), f'Upload IFCNetCorePng.zip to {DRIVE_ZIP.parent} first.'
print('zip   :', DRIVE_ZIP, '(', DRIVE_ZIP.stat().st_size // (1024*1024), 'MB)')
print('drive :', DRIVE_OUT)

In [ ]:
# Unzip renders (skips if already unpacked)
RENDERS_ROOT.mkdir(parents=True, exist_ok=True)
!unzip -q -n "$DRIVE_ZIP" -d "$RENDERS_ROOT"
n_png = sum(1 for _ in RENDERS_ROOT.rglob('*.png'))
print(f'unzipped: {n_png} PNGs')

In [ ]:
# Generate metadata.json from the unzipped tree (idempotent)
import json
rows = []
for class_dir in sorted(RENDERS_ROOT.iterdir()):
    if not class_dir.is_dir(): continue
    for split_dir in sorted(class_dir.iterdir()):
        if not split_dir.is_dir(): continue
        obj_ids = sorted({p.stem.rsplit('.', 1)[0] for p in split_dir.glob('*.png')})
        for oid in obj_ids:
            views = sorted(split_dir.glob(f'{oid}.*.png'))
            rows.append({'obj_id': oid, 'ifc_class': class_dir.name,
                         'split': split_dir.name, 'num_renders': len(views)})
METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
METADATA_PATH.write_text(json.dumps(rows, indent=2))
print(f'{len(rows)} objects, {len({r["ifc_class"] for r in rows})} classes')

In [ ]:
# Shared helpers
import json, shutil, subprocess
from dataclasses import dataclass
from tqdm import tqdm
import numpy as np


@dataclass(frozen=True)
class ObjectEntry:
    obj_id: str
    ifc_class: str
    split: str
    views: list


def load_entries():
    out = []
    for r in json.loads(METADATA_PATH.read_text()):
        d = RENDERS_ROOT / r['ifc_class'] / r['split']
        out.append(ObjectEntry(r['obj_id'], r['ifc_class'], r['split'],
                               sorted(d.glob(f"{r['obj_id']}.*.png"))))
    return out


def filter_pending(entries, out_dir):
    return [e for e in entries if not (out_dir / f'{e.obj_id}.npy').exists()]


def zip_and_upload(features_subdir, archive_name, drive_out=DRIVE_OUT):
    src = features_subdir / 'colorless'
    n = sum(1 for _ in src.glob('*.npy'))
    if n == 0:
        print(f'  [warn] {src} empty — nothing to zip'); return None
    local = Path('/content') / archive_name
    if local.exists(): local.unlink()
    subprocess.run(['zip', '-qr', str(local), 'colorless'], cwd=str(features_subdir), check=True)
    drive_out.mkdir(parents=True, exist_ok=True)
    drive_zip = drive_out / archive_name
    shutil.copy2(local, drive_zip)
    print(f'  ✓ zipped {n} files → {drive_zip}  ({drive_zip.stat().st_size/1024/1024:.1f} MB)')
    return drive_zip


ENTRIES = load_entries()
print(f'{len(ENTRIES)} object entries')

## Install + run SigLIP 2 (~1.2 h on T4 for 7930 objects)

In [ ]:
!pip install -q "transformers>=4.49" pillow

In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor

SIGLIP_MODEL_ID = 'google/siglip2-large-patch16-512'
SIGLIP_OUT = FEATURES_ROOT / 'siglip2_large_16_512' / 'colorless'
SIGLIP_OUT.mkdir(parents=True, exist_ok=True)


def _to_tensor(out):
    """Unwrap whatever get_image_features() returns into a (N, D) tensor.

    Different transformers versions return different types here:
      - bare Tensor (older / direct image tower)
      - BaseModelOutputWithPooling (.pooler_output)
      - CLIPVisionModelOutput-style (.image_embeds)
    """
    if isinstance(out, torch.Tensor):
        return out
    for attr in ('image_embeds', 'pooler_output', 'last_hidden_state'):
        if hasattr(out, attr):
            t = getattr(out, attr)
            return t.mean(dim=1) if attr == 'last_hidden_state' and t.ndim == 3 else t
    raise TypeError(f'cannot extract tensor from {type(out).__name__}')


pending = filter_pending(ENTRIES, SIGLIP_OUT)
print(f'[siglip] {len(pending)} / {len(ENTRIES)} pending')

if pending:
    siglip = AutoModel.from_pretrained(SIGLIP_MODEL_ID).cuda().eval()
    siglip_proc = AutoProcessor.from_pretrained(SIGLIP_MODEL_ID)

    # smoke: print the actual tensor shape before the long loop
    with torch.no_grad():
        _pil = Image.open(pending[0].views[0]).convert('RGB')
        _in = siglip_proc(images=[_pil], return_tensors='pt')['pixel_values'].cuda()
        _raw = siglip.get_image_features(_in)
        _t = _to_tensor(_raw)
    print(f'[siglip] smoke: type={type(_raw).__name__}  shape={tuple(_t.shape)}')

    for e in tqdm(pending, desc='siglip'):
        try:
            pil_list = [Image.open(v).convert('RGB') for v in e.views]
            inputs = siglip_proc(images=pil_list, padding=True, truncation=True, return_tensors='pt')
            with torch.no_grad():
                raw = siglip.get_image_features(inputs['pixel_values'].cuda())
                feats = _to_tensor(raw)
            vec = feats.mean(dim=0).cpu().numpy().astype(np.float32)
            np.save(SIGLIP_OUT / f'{e.obj_id}.npy', vec)
        except Exception as exc:
            print(f'  [error] {e.obj_id}: {exc}')

    del siglip, siglip_proc
    torch.cuda.empty_cache()

print('done:', len(list(SIGLIP_OUT.glob('*.npy'))), 'embeddings written')

In [ ]:
zip_and_upload(FEATURES_ROOT / 'siglip2_large_16_512',
               archive_name='siglip_ifcnet_colorless.zip')